In [7]:
import numpy as np
import torch


### Реализация функций потерь.

In [8]:
class MSELoss:
    """
    Функция потерь Mean Squared Error (MSELoss).
    """

    def forward(self, pred: np.ndarray, target: np.ndarray) -> float:
        """
        Прямой проход через MSELoss.

        pred   : выход с последнего слоя сети, shape (bs, ...)
        target : целевые значения, shape (bs, ...)
        """
        self.E = target - pred
        R = self.E ** 2

        return np.sum(R) / R.size

    def backward(self) -> np.ndarray:
        """
        Обратный проход через MSELoss.
        """
        return -2 * self.E / self.E.size

    def __call__(self, pred: np.ndarray, target: np.ndarray) -> float:
        """Вызов forward при использовании экземпляра как функции."""
        return self.forward(pred, target)

In [9]:
class CrossEntropyLoss:
    """
    Функция потерь Cross-Entropy для задач многоклассовой классификации.
    Реализация: Softmax + Cross-Entropy Loss.
    """

    def forward(self, pred: np.ndarray, target: np.ndarray) -> float:
        """
        Прямой проход через Cross-Entropy loss.

        pred   : выход с последнего слоя сети, shape (bs, C)
        target : one-hot метки классов, shape (bs, C)
        """
        self.batch_size = pred.shape[0]
        # Сохраняю target для backward.
        self.target = target

        # Вычитаю максимум из exp. для численно стабильных вычислений.
        logits_max = pred.max(axis=1, keepdims=True)
        exp_pred = np.exp(pred - logits_max)

        # Считаю и сохраняю Softmax для backward.
        self.softmax = exp_pred / exp_pred.sum(axis=1, keepdims=True)

        # log-sum-exp
        log_sum_exp = np.log(exp_pred.sum(axis=1, keepdims=True))

        # Считаю функцию потерь: Softmax + Cross-Entropy Loss.
        loss = (
            -np.sum(target * pred, axis=1, keepdims=True)
            + logits_max
            + log_sum_exp
        )

        return loss.mean()

    def backward(self) -> np.ndarray:
        """
        Обратный проход через: Softmax + Cross-Entropy Loss.
        """
        return (self.softmax - self.target) / self.batch_size

    def __call__(self, pred: np.ndarray, target: np.ndarray) -> float:
        return self.forward(pred, target)

In [10]:
class BCEWithLogitsLoss:
    """
    Функция потерь Binary Cross-Entropy для задач бинарной классификации.
    Реализация: Sigmoid + Binary Cross-Entropy Loss.
    """
    
    def forward(self, pred: np.ndarray, target: np.ndarray) -> float:
        """
        Прямой проход через Binary Cross-Entropy Loss.

        pred   : выход с последнего слоя сети, shape (bs, C)
        target : метки классов, shape (bs, C)
        """
        self.N = pred.size
        # Сохраняю target для backward.
        self.target = target

        # Считаю и сохраняю Sigmoid для backward.
        self.sigm = 1 / (1 + np.exp(-pred))

        r = self.relu(-pred)

        # Считаю функцию потерь: Sigmoid + Binary Cross-Entropy Loss.
        bceloss = (
            (1 - target) * pred
            + r
            + np.log(np.exp(-r) + np.exp(-(pred + r)))
        )

        return bceloss.mean()

    def backward(self) -> np.ndarray:
        """
        Обратный проход через: Sigmoid + Binary Cross-Entropy Loss.
        """
        return (self.sigm - self.target) / self.N

    def relu(self, pred: np.ndarray) -> np.ndarray:
        """
        Функция Rectified Linear Unit.
        """
        return np.maximum(0, pred)

    def __call__(self, pred: np.ndarray, target: np.ndarray) -> float:
        return self.forward(pred, target)

### Проверка реализованных функций.

In [11]:
pred = torch.randn(16, 10, dtype=torch.float32, requires_grad=True)
target = torch.randn(16, 10).softmax(dim=1)

""" ======= PyTorch implementation ======= """
loss_func = torch.nn.CrossEntropyLoss()

# forward
l = loss_func(pred, target)
# backward
l.backward()

""" ======= Numpy implementation ======= """
my_loss_func = CrossEntropyLoss()

# forward
my_loss = my_loss_func(pred.detach().numpy(), target.numpy())
# backward
my_grad = my_loss_func.backward()

print("PyTorch CrossEntropyLoss =", l.item())
print("My CELoss implementation =", my_loss.item())
print("'PyTorch CELoss' == 'My CELoss' ==>", np.allclose(l.detach().numpy(), my_loss))
print("'PyTorch grad' == 'My grad'     ==>", np.allclose(pred.grad.detach().numpy(), my_grad))



PyTorch CrossEntropyLoss = 2.725098133087158
My CELoss implementation = 2.725098133087158
'PyTorch CELoss' == 'My CELoss' ==> True
'PyTorch grad' == 'My grad'     ==> True


In [13]:
pred = torch.randn(16, 10, dtype=torch.float32, requires_grad=True)
target = torch.randn(16, 10).softmax(dim=1)

""" ======= PyTorch implementation ======= """
loss_func = torch.nn.BCEWithLogitsLoss()

# forward
l = loss_func(pred, target)
# backward
l.backward()

""" ======= Numpy implementation ======= """
my_loss_func = BCEWithLogitsLoss()

# forward
my_loss = my_loss_func(pred.detach().numpy(), target.numpy())
# backward
my_grad = my_loss_func.backward()

print("PyTorch BCEWithLogitsLoss =", l.item())
print("My BCELoss implementation =", my_loss.item())
print("'PyTorch CELoss' == 'My CELoss' ==>", np.allclose(l.detach().numpy(), my_loss))
print("'PyTorch grad' == 'My grad'     ==>", np.allclose(pred.grad.detach().numpy(), my_grad))



PyTorch BCEWithLogitsLoss = 0.814052939414978
My BCELoss implementation = 0.814052939414978
'PyTorch CELoss' == 'My CELoss' ==> True
'PyTorch grad' == 'My grad'     ==> True
